In [ ]:
import numpy as np
from scipy.optimize import minimize



---


In [ ]:
# 1. LOAD OR DEFINE DATA


---


In [ ]:
# Define the data directly based on your input text files
Q = np.array([
    [1.023083240688652973e+00, 1.151266171855282794e+00, 1.520494378714184247e+00, 7.763137127990872299e-01, 8.433711589310854961e-01],
    [1.151266171855282794e+00, 2.090771156380078377e+00, 2.354815529150700470e+00, 1.367747789846088136e+00, 1.065255192243313553e+00],
    [1.520494378714184247e+00, 2.354815529150700470e+00, 3.117460834754574694e+00, 1.752273330046332456e+00, 1.566817467002336928e+00],
    [7.763137127990872299e-01, 1.367747789846088136e+00, 1.752273330046332456e+00, 1.226396596416716722e+00, 9.570741754007232593e-01],
    [8.433711589310854961e-01, 1.065255192243313553e+00, 1.566817467002336928e+00, 9.570741754007232593e-01, 1.202578962788080075e+00]
])

b = np.array([
    3.651808011323913750e-01,
    1.953975998715540285e-01,
    5.948658710060528154e-01,
    4.353131559506445702e-01,
    2.999915037451498234e-01
])

c = 2.094161212362195545e-01



Alternatively, if running locally where the files exist:
Q = np.loadtxt('Q.txt')
b = np.loadtxt('b.txt')
c = np.loadtxt('c.txt')


---


In [ ]:
# 2. DEFINE OBJECTIVE FUNCTION & GRADIENT


---


In [ ]:
def f(x):
    return x.T @ Q @ x + b.T @ x + c

def grad_f(x):
    # Note: grad(x^T Q x) = 2 * Q * x because Q is symmetric
    return 2 * Q @ x + b



---


In [ ]:
# PART (a): CUSTOM GRADIENT DESCENT WITH ARMIJO


---


In [ ]:
def gradient_descent_armijo(x0, eps_tol=1e-3, k_max=5000):
    x = x0.copy()
    
    # Armijo parameters chosen per instructions
    alpha_init = 1.0  # Must be > 0.5
    epsilon = 0.15    # Must be in [0.1, 0.2]
    beta = 0.85       # Must be in [0.8, 0.9]
    
    k = 0
    while k < k_max:
        g = grad_f(x)
        norm_g = np.linalg.norm(g)
        
        # Check termination condition
        if norm_g <= eps_tol:
            break
            
        # Search direction
        d = -g
        
        # Inexact line search using Armijo's rule
        alpha = alpha_init
        while True:
            # Step (2) Condition
            if f(x + alpha * d) <= f(x) + epsilon * alpha * (g.T @ d):
                break  # Accept alpha_k = alpha
            else:
                alpha = beta * alpha  # Backtracking step (3)
                
        # Update point
        x = x + alpha * d
        k += 1
        
    return x, f(x), np.linalg.norm(grad_f(x)), k

# Initialize at zero vector
x0 = np.zeros(5) 
x_k, f_xk, tol_reached, num_iter = gradient_descent_armijo(x0)

print("=== PART (a): Custom Gradient Descent ===")
print(f"(i)   Termination point x_k:\n{x_k}")
print(f"(ii)  Objective value f(x_k): {f_xk:.6f}")
print(f"(iii) Tolerance reached: {tol_reached:.2e}")
print(f"(iv)  Number of iterations k: {num_iter}\n")



---


In [ ]:
# PART (b): FIRST-ORDER NECESSARY CONDITIONS (FONCs)


---


In [ ]:
# 2 * Q * x* + b = 0  ->  x* = -0.5 * Q^-1 * b
x_fonc = np.linalg.solve(2 * Q, -b)
f_fonc = f(x_fonc)

print("=== PART (b): FONC Analytical Solution ===")
print(f"Optimal x*:\n{x_fonc}")
print(f"Optimal f(x*): {f_fonc:.6f}\n")



---


In [ ]:
# PART (c): SCI PY OPTIMIZATION SOLVER


---


In [ ]:
res = minimize(f, x0, method='BFGS')
scipy_grad_norm = np.linalg.norm(grad_f(res.x))

print("=== PART (c): SciPy minimize Comparison ===")
print(f"SciPy x*:\n{res.x}")
print(f"SciPy f(x*): {res.fun:.6f}")
print(f"SciPy tolerance reached (||∇f(x*)||): {scipy_grad_norm:.2e}")
print(f"SciPy number of iterations: {res.nit}")
print(f"Success status: {res.success}")
